# PRODUCTION FRAMEWORK & MLOPS (End-to-End)

This notebook simulates the production environment (MLOps) for the **14-day Strategic Replenishment System**. 

It consolidates the entire machine learning lifecycle into functional blocks:
1. **Data Preprocessing Pipeline**: Clean and robust feature transformation.
2. **Model Retraining Pipeline**: Continuous Integration (CI) to ensure the model adapts to new sales trends.
3. **Daily Inference & Business Logic**: The "Brain" behind the Alerting Suite.

**Strategic Prioritization**: Instead of flooding the supply chain team with alerts, we apply a **Financial Risk Score** (Probability * Price * Sales Velocity). This transforms raw data into a highly actionable Top 20 list, solving the alert fatigue problem while maximizing revenue protection.

In [3]:
# ----------------------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

# Define global paths
PROJECT_PATH = '/Users/rober/retail-stockout-risk-scoring'
DATA_PATH = PROJECT_PATH + '/02_Data/03_Working/df_feature_selection.pickle'
MODEL_PATH = PROJECT_PATH + '/04_Models/champion_xgb_production.pkl'

# ----------------------------------------------------------------------------
# 2. DATA PREPROCESSING PIPELINE
# ----------------------------------------------------------------------------
def preprocess_pipeline(df_raw, is_training=False):
    """
    Standardizes data for the model. 
    Ensures that ONLY selected features reach the engine.
    """
    df = df_raw.copy()
    
    # In production, we drop the target if we are inferencing
    if not is_training and 'target' in df.columns:
        df = df.drop(columns=['target'])
        
    return df

# ----------------------------------------------------------------------------
# 3. MODEL RETRAINING PIPELINE (CI)
# ----------------------------------------------------------------------------
def retrain_model(historical_data_path, output_model_path):
    print("\n[SYSTEM] Initiating Model Retraining Pipeline...")
    df_raw = pd.read_pickle(historical_data_path)
    
    X = df_raw.drop(columns=['target'])
    y = df_raw['target']
    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    weight_ratio = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
    
    # Using the validated 14-day architecture
    model = XGBClassifier(
        n_estimators=400, 
        learning_rate=0.05,
        max_depth=4,
        scale_pos_weight=weight_ratio,
        eval_metric='logloss',
        random_state=42
    )
    model.fit(X_train, y_train)
    
    val_probs = model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_probs)
    print(f"[SYSTEM] Retraining complete. Validation AUC: {val_auc:.4f}")
    
    # Quality Gate (Realistic threshold)
    if 0.80 < val_auc < 0.98: # If it's > 0.98, it's probably leakage again!
        joblib.dump(model, output_model_path)
        print(f"[SUCCESS] Model deployed to: {output_model_path}")
    else:
        print("[ERROR] Deployment aborted: Signal is too weak or suspiciously perfect.")

# ----------------------------------------------------------------------------
# 4. INFERENCE & BUSINESS LOGIC PIPELINE
# ----------------------------------------------------------------------------
def run_daily_inference(daily_data_path, model_path, top_n_alerts=15):
    print("\n[SYSTEM] Initiating Daily Inference & Alert Pipeline...")
    df_raw = pd.read_pickle(daily_data_path)
    
    # Preprocess
    df_scoring = preprocess_pipeline(df_raw, is_training=False)
    
    # Load model
    model = joblib.load(model_path)
    
    # 1. Scoring
    df_raw['stockout_probability'] = model.predict_proba(df_scoring)[:, 1]
    
    # 2. Business Logic: Financial Risk Score
    # Risk = Probability * Market Price * Sales Momentum
    df_raw['financial_risk'] = df_raw['stockout_probability'] * df_raw['price'] * df_raw['units_sold']
    
    # 3. Filtering: High Confidence or High Financial Impact
    # We use 0.75 as the professional confidence threshold
    final_alerts = df_raw[df_raw['stockout_probability'] > 0.70].sort_values(
        by='financial_risk', ascending=False
    ).head(top_n_alerts)
    
    # 4. Simulation Output
    print(f"\n" + "="*60)
    print(f" 📱 TELEGRAM SUITE: TOP {top_n_alerts} STRATEGIC ALERTS (14-DAY) ")
    print("="*60)
    
    for index, row in final_alerts.iterrows():
        # Clean simulation of business impact
        impact = "HIGH" if row['financial_risk'] > 0.6 else "MEDIUM"
        print(f"🚨 ALERT | {impact} IMPACT - Product ID: {index}")
        print(f"    Risk Probability: {row['stockout_probability']*100:.1f}%")
        print(f"    Business Signal: Price {row['price']:.2f} | Sales Vel. {row['units_sold']:.2f}")
        print("-" * 60)
        
    return final_alerts

# ----------------------------------------------------------------------------
# 5. EXECUTION
# ----------------------------------------------------------------------------
if __name__ == "__main__":
    retrain_model(DATA_PATH, MODEL_PATH)
    daily_alerts = run_daily_inference(DATA_PATH, MODEL_PATH)


[SYSTEM] Initiating Model Retraining Pipeline...
[SYSTEM] Retraining complete. Validation AUC: 0.9085
[SUCCESS] Model deployed to: /Users/rober/retail-stockout-risk-scoring/04_Models/champion_xgb_production.pkl

[SYSTEM] Initiating Daily Inference & Alert Pipeline...

 📱 TELEGRAM SUITE: TOP 15 STRATEGIC ALERTS (14-DAY) 
🚨 ALERT | MEDIUM IMPACT - Product ID: 11814
    Risk Probability: 72.4%
    Business Signal: Price 0.98 | Sales Vel. 0.18
------------------------------------------------------------
🚨 ALERT | MEDIUM IMPACT - Product ID: 11082
    Risk Probability: 71.5%
    Business Signal: Price 0.99 | Sales Vel. 0.18
------------------------------------------------------------
🚨 ALERT | MEDIUM IMPACT - Product ID: 24470
    Risk Probability: 70.9%
    Business Signal: Price 0.98 | Sales Vel. 0.16
------------------------------------------------------------
🚨 ALERT | MEDIUM IMPACT - Product ID: 20144
    Risk Probability: 75.7%
    Business Signal: Price 0.99 | Sales Vel. 0.15
------

In [4]:
# ----------------------------------------------------------------------------
# 6. SUITE QUALITY CERTIFICATION (EXECUTIVE SUMMARY)
# ----------------------------------------------------------------------------
print("\n" + "="*60)
print(" 📈 SUITE CERTIFICATION REPORT ")
print("="*60)

# Cargamos los datos de nuevo para el conteo global (o usamos la ruta que ya tenemos)
df_final_check = pd.read_pickle(DATA_PATH)
model_final = joblib.load(MODEL_PATH)

# Calculamos probabilidades para todo el dataset para el reporte ejecutivo
total_probs = model_final.predict_proba(df_final_check.drop(columns=['target'], errors='ignore'))[:, 1]

total_products = len(df_final_check)
risk_detected = (total_probs > 0.50).sum()
critical_alerts = (total_probs > 0.75).sum()
top_impact_alerts = len(daily_alerts) # Esta variable sí está definida en tu bloque principal

print(f"✅ Total Products Scanned: {total_products}")
print(f"⚠️ Products with moderate risk (>50%): {risk_detected} ({risk_detected/total_products:.1%})")
print(f"🚨 Critical stockouts identified (>75%): {critical_alerts}")
print(f"🎯 Final prioritized alerts sent to Telegram: {top_impact_alerts}")
print(f"💰 Average Risk Probability in Top 15: {daily_alerts['stockout_probability'].mean():.2%}")
print("="*60)
print("[STATUS] System certified. Signal-to-noise ratio is optimal.")


 📈 SUITE CERTIFICATION REPORT 
✅ Total Products Scanned: 51170
⚠️ Products with moderate risk (>50%): 12523 (24.5%)
🚨 Critical stockouts identified (>75%): 4843
🎯 Final prioritized alerts sent to Telegram: 15
💰 Average Risk Probability in Top 15: 78.13%
[STATUS] System certified. Signal-to-noise ratio is optimal.
